In [4]:
# Core
import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

# Data
import pandas as pd
import numpy as np
import requests

# Database
from sqlalchemy import create_engine, text, inspect
import psycopg2

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 50)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print('✅ All imports loaded successfully!')

✅ All imports loaded successfully!


In [3]:
import pandas as pd

df = pd.read_csv('../data_source/medical-appointments-no-show-en.csv')
df.head()


,specialty,appointment_time,gender,appointment_date,no_show,no_show_reason,disability,date_of_birth,entry_service_date,city,...,over_60_years_old,patient_needs_companion,average_temp_day,average_rain_day,max_temp_day,max_rain_day,rainy_day_before,storm_day_before,rain_intensity,heat_intensity
0,physiotherapy,13:20,M,09/09/2021,yes,surto,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
1,psychotherapy,13:20,M,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
2,speech therapy,13:20,F,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
3,physiotherapy,13:20,F,09/09/2021,no,NaN,NaN,NaN,NaN,NaN,...,0,0,20.75,0.01,23.7,0.2,1,1,no_rain,mild
4,physiotherapy,14:00,M,09/09/2021,no,NaN,motor,10/10/1954,5/2/2020,B. CAMBORIU,...,1,1,20.75,0.01,23.7,0.2,1,1,no_rain,mild


In [7]:
#show column name
for col in df.columns:
    print(col)

specialty
appointment_time
gender
appointment_date
no_show
no_show_reason
disability
date_of_birth
entry_service_date
city
icd
appointment_month
appointment_year
appointment_shift
age
under_12_years_old
over_60_years_old
patient_needs_companion
average_temp_day
average_rain_day
max_temp_day
max_rain_day
rainy_day_before
storm_day_before
rain_intensity
heat_intensity


In [8]:
#rename the column icd to dianosis_code
df.rename(columns={'icd': 'diagnosis_code'}, inplace=True)

In [9]:
#show column name
for col in df.columns:
    print(col)

specialty
appointment_time
gender
appointment_date
no_show
no_show_reason
disability
date_of_birth
entry_service_date
city
diagnosis_code
appointment_month
appointment_year
appointment_shift
age
under_12_years_old
over_60_years_old
patient_needs_companion
average_temp_day
average_rain_day
max_temp_day
max_rain_day
rainy_day_before
storm_day_before
rain_intensity
heat_intensity


In [ ]:
# Shape & info
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum().sort_values(ascending=False)}")
print(f"\nDuplicates: {df.duplicated().sum()}")

Shape: (49593, 26)

Missing values:
no_show_reason             47856
icd                        38876
age                        10350
date_of_birth              10321
specialty                   7454
city                        5181
entry_service_date          5155
disability                  5137
average_temp_day            1016
average_rain_day            1016
max_rain_day                1016
max_temp_day                1016
no_show                        0
appointment_time               0
appointment_date               0
gender                         0
appointment_year               0
appointment_month              0
patient_needs_companion        0
over_60_years_old              0
under_12_years_old             0
appointment_shift              0
rainy_day_before               0
storm_day_before               0
rain_intensity                 0
heat_intensity                 0
dtype: int64

Duplicates: 2921


In [10]:
# 1. Parse dates
df['appointment_date'] = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y', errors='coerce')
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], format='%d/%m/%Y', errors='coerce')
df['entry_service_date'] = pd.to_datetime(df['entry_service_date'], format='%d/%m/%Y', errors='coerce')
# 2. Clean disability column (space → NaN)
df['disability'] = df['disability'].replace(' ', np.nan)
# 3. Encode target as binary
df['no_show_binary'] = (df['no_show'] == 'yes').astype(int)
# 4. Cap age outliers (e.g., age > 105 → NaN)
df.loc[df['age'] > 105, 'age'] = np.nan
# 5. Drop duplicates
df.drop_duplicates(inplace=True)
print(f"✅ Cleaned! Shape: {df.shape}")
print(f"No-show rate: {df['no_show_binary'].mean()*100:.1f}%")

✅ Cleaned! Shape: (46672, 27)
No-show rate: 10.0%


---
# 📊 Exploratory Data Analysis (EDA)

## Business Impact of No-Shows

Understanding the financial and operational cost of missed appointments.
Each no-show represents wasted provider time, lost revenue, and delayed care for other patients.


In [11]:
# === OVERALL NO-SHOW RATE ===
total = len(df)
no_shows = df['no_show_binary'].sum()
show_ups = total - no_shows
rate = no_shows / total * 100

print("=" * 55)
print("   📋 NO-SHOW SUMMARY")
print("=" * 55)
print(f"   Total Appointments:    {total:>10,}")
print(f"   Showed Up:             {show_ups:>10,}")
print(f"   No-Shows:              {no_shows:>10,}")
print(f"   No-Show Rate:          {rate:>9.1f}%")
print("=" * 55)

fig = px.pie(
    values=[show_ups, no_shows],
    names=['Showed Up', 'No-Show'],
    title=f'Overall No-Show Rate ({rate:.1f}%)',
    color_discrete_sequence=['#2A9D8F', '#E76F51'],
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label+value')
fig.update_layout(height=400)
fig.show()


   📋 NO-SHOW SUMMARY
   Total Appointments:        46,672
   Showed Up:                 41,994
   No-Shows:                   4,678
   No-Show Rate:               10.0%


## No-Show by Specialty

Which specialties suffer the most from missed appointments?


In [12]:
# === NO-SHOW RATE BY SPECIALTY ===
spec = df.groupby('specialty').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
spec['rate'] = (spec['no_shows'] / spec['total'] * 100).round(1)
spec = spec.sort_values('rate', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    name='Showed Up', x=spec['specialty'], y=spec['total'] - spec['no_shows'],
    marker_color='#2A9D8F'
))
fig.add_trace(go.Bar(
    name='No-Show', x=spec['specialty'], y=spec['no_shows'],
    marker_color='#E76F51'
))
fig.update_layout(
    barmode='stack', title='Appointments by Specialty (Show vs No-Show)',
    xaxis_title='Specialty', yaxis_title='Count', height=450
)
fig.show()

# Rate comparison
fig2 = px.bar(
    spec, x='specialty', y='rate',
    title='No-Show Rate by Specialty (%)',
    color='rate', color_continuous_scale='RdYlGn_r',
    text='rate'
)
fig2.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig2.update_layout(height=400, coloraxis_showscale=False, yaxis_title='No-Show Rate (%)')
fig2.show()

print("\nNo-show rate by specialty:")
for _, row in spec.iterrows():
    print(f"  {row['specialty']:<25} {row['rate']:>5.1f}%  ({row['no_shows']:,} / {row['total']:,})")



No-show rate by specialty:
  sem especialidade          22.8%  (13 / 57)
  physiotherapy              10.5%  (930 / 8,833)
  speech therapy              9.7%  (1,109 / 11,443)
  psychotherapy               9.6%  (1,075 / 11,204)
  occupational therapy        8.7%  (550 / 6,334)
  pedagogo                    7.8%  (154 / 1,965)
  assist                      6.7%  (13 / 194)
  enf                         6.1%  (42 / 694)


## No-Show Reasons

What reasons do patients give for missing appointments? (Only available for a subset of no-shows)


In [16]:
# Categorize no-show reasons into English groups
def categorize_reason(reason):
    if pd.isna(reason):
        return np.nan
    r = reason.lower().strip()
    
    # Illness
    if any(w in r for w in ['doente', 'doença', 'gripe', 'gripado', 'gripada', 'febre', 'febril',
                             'resfriado', 'resfriada', 'virose', 'tosse', 'diarreia', 'diarr',
                             'vômito', 'v\xf4mito', 'enjoada', 'náuseas', 'mal esta', 'ruim',
                             'sintoma', 'convuls', 'infec', 'amidalite', 'conjuntivite',
                             'dor de', 'com dor', 'pressão alta', 'cólica', 'pernas inchadas',
                             'está doente', 'esta doente', 'estava doente', 'passou mal',
                             'se sentindo mal', 'tontura', 'sonol', 'indisposta']):
        return 'Illness'
    
    # COVID-related
    if any(w in r for w in ['covid', 'quarentena', 'isolamento', 'pandemia', 'vacina']):
        return 'COVID / Vaccine'
    
    # Transportation
    if any(w in r for w in ['transporte', 'carro', 'moto', 'pneu', 'gasolina', 'carona',
                             'ônibus', 'locomoção', 'trânsito', 'passagem', 'bike']):
        return 'Transportation'
    
    # Cancelled / Rescheduled
    if any(w in r for w in ['desmarcado', 'desmarcou', 'cancelad', 'cancelou', 'remarcado',
                             'remarcou', 'reagendado', 'remarc', 'mudou data', 'mudou horário',
                             'troca de horário', 'troca de v', 'alteração']):
        return 'Cancelled / Rescheduled'
    
    # Other medical
    if any(w in r for w in ['cirurgia', 'internado', 'internada', 'hospitalizado', 'consulta',
                             'medico', 'médico', 'dentista', 'exame', 'alta', 'recebeu alta',
                             'perícia', 'pericia', 'atestado', 'relatório', 'prótese',
                             'órtese', 'botox', 'odontol']):
        return 'Other Medical Appointment'
    
    # Family issues
    if any(w in r for w in ['mãe', 'mae', 'pai', 'irmã', 'irmão', 'irm', 'familiar',
                             'esposa', 'filha', 'neta', 'sogro', 'faleceu', 'falecimento',
                             'velório', 'morte']):
        return 'Family Issue'
    
    # Work / School
    if any(w in r for w in ['trabalho', 'escola', 'aula', 'curso', 'entrevista', 'evento',
                             'compromisso', 'apresentação', 'prova', 'banco de horas']):
        return 'Work / School'
    
    # Weather
    if any(w in r for w in ['chuva', 'chovendo', 'frio', 'mau tempo', 'lama']):
        return 'Weather'
    
    # Gave up / Left
    if any(w in r for w in ['desist', 'desligad', 'se mudou', 'transferi']):
        return 'Dropped Out / Moved'
    
    # Traveling
    if any(w in r for w in ['viaj', 'viagem']):
        return 'Traveling'
    
    # Accident / Injury
    if any(w in r for w in ['caiu', 'acidente', 'machucou', 'mordida', 'fraturou', 'quebrou']):
        return 'Accident / Injury'
    
    return 'Other'

df['no_show_reason_category'] = df['no_show_reason'].apply(categorize_reason)

# Show distribution
cats = df[df['no_show'] == 'yes']['no_show_reason_category'].dropna().value_counts()
print("No-show reasons by category:")
for cat, count in cats.items():
    print(f"  {cat:<30} {count:>5,}")


No-show reasons by category:
  Illness                          510
  Transportation                   270
  Cancelled / Rescheduled          214
  Other Medical Appointment        196
  Other                            170
  COVID / Vaccine                   77
  Work / School                     58
  Family Issue                      41
  Dropped Out / Moved               39
  Traveling                         34
  Weather                           17
  Accident / Injury                 11


In [17]:
# === NO-SHOW REASONS BY CATEGORY ===
cats = df[df['no_show'] == 'yes']['no_show_reason_category'].dropna().value_counts()

fig = px.bar(
    x=cats.values, y=cats.index, orientation='h',
    title='No-Show Reasons by Category (English)',
    labels={'x': 'Count', 'y': 'Reason Category'},
    color=cats.values, color_continuous_scale='Reds'
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
fig.show()



## No-Show by Gender


In [18]:
# === NO-SHOW BY GENDER ===
gender = df.groupby('gender').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
gender['rate'] = (gender['no_shows'] / gender['total'] * 100).round(1)

fig = px.bar(
    gender, x='gender', y='rate',
    title='No-Show Rate by Gender',
    color='gender', color_discrete_sequence=['#3A7CA5', '#E07A5F', '#6BAA75'],
    text='rate'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=400, showlegend=False, yaxis_title='No-Show Rate (%)')
fig.show()



## No-Show by Age Group

Are certain age groups more likely to miss appointments?


In [19]:
# === NO-SHOW BY AGE GROUP ===
bins = [0, 12, 18, 30, 45, 60, 80, 120]
labels = ['0-12', '13-18', '19-30', '31-45', '46-60', '61-80', '80+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

age_ns = df.groupby('age_group', observed=True).agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
age_ns['rate'] = (age_ns['no_shows'] / age_ns['total'] * 100).round(1)

fig = px.bar(
    age_ns, x='age_group', y='rate',
    title='No-Show Rate by Age Group',
    color='rate', color_continuous_scale='RdYlGn_r',
    text='rate'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=400, coloraxis_showscale=False,
                  xaxis_title='Age Group', yaxis_title='No-Show Rate (%)')
fig.show()


## No-Show by Time Factors
Does the month, year, or time of day affect no-show rates?

In [21]:
# === NO-SHOW BY MONTH ===
month_order = ['jan', 'feb', 'mar', 'apr', 'may', 'june', 'july', 'aug', 'sept', 'oct', 'nov', 'dec']
month_ns = df.groupby('appointment_month').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
month_ns['rate'] = (month_ns['no_shows'] / month_ns['total'] * 100).round(1)
month_ns['appointment_month'] = pd.Categorical(month_ns['appointment_month'], categories=month_order, ordered=True)
month_ns = month_ns.sort_values('appointment_month')

fig = px.line(
    month_ns, x='appointment_month', y='rate', markers=True,
    title='No-Show Rate by Month',
    labels={'appointment_month': 'Month', 'rate': 'No-Show Rate (%)'}
)
fig.update_layout(height=400)
fig.show()

# === NO-SHOW BY SHIFT ===
shift_ns = df.groupby('appointment_shift').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
shift_ns['rate'] = (shift_ns['no_shows'] / shift_ns['total'] * 100).round(1)

fig2 = px.bar(
    shift_ns, x='appointment_shift', y='rate',
    title='No-Show Rate: Morning vs Afternoon',
    color='appointment_shift', color_discrete_sequence=['#E9C46A', '#264653'],
    text='rate'
)
fig2.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig2.update_layout(height=400, showlegend=False, yaxis_title='No-Show Rate (%)')
fig2.show()

# === TREND BY YEAR ===
year_ns = df.groupby('appointment_year').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
year_ns['rate'] = (year_ns['no_shows'] / year_ns['total'] * 100).round(1)

fig3 = px.bar(
    year_ns, x='appointment_year', y='rate',
    title='No-Show Rate by Year (Trend)',
    text='rate', color='rate', color_continuous_scale='RdYlGn_r'
)
fig3.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig3.update_layout(height=400, coloraxis_showscale=False, yaxis_title='No-Show Rate (%)')
fig3.show()


## No-Show by Weather Conditions

Does rain, temperature, or storms affect whether patients show up?


In [22]:
# === NO-SHOW BY RAIN INTENSITY ===
rain_ns = df.groupby('rain_intensity').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
rain_ns['rate'] = (rain_ns['no_shows'] / rain_ns['total'] * 100).round(1)

fig = px.bar(
    rain_ns, x='rain_intensity', y='rate',
    title='No-Show Rate by Rain Intensity',
    color='rain_intensity', color_discrete_sequence=['#E9C46A', '#81B29A', '#3D405B', '#264653'],
    text='rate'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=400, showlegend=False, yaxis_title='No-Show Rate (%)')
fig.show()

# === NO-SHOW BY HEAT INTENSITY ===
heat_ns = df.groupby('heat_intensity').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
heat_ns['rate'] = (heat_ns['no_shows'] / heat_ns['total'] * 100).round(1)

fig2 = px.bar(
    heat_ns, x='heat_intensity', y='rate',
    title='No-Show Rate by Heat Intensity',
    color='heat_intensity', color_discrete_sequence=['#3D405B', '#264653', '#81B29A', '#E9C46A', '#E76F51'],
    text='rate'
)
fig2.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig2.update_layout(height=400, showlegend=False, yaxis_title='No-Show Rate (%)')
fig2.show()

# === STORM DAY BEFORE ===
storm = df.groupby('storm_day_before').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
storm['rate'] = (storm['no_shows'] / storm['total'] * 100).round(1)
storm['storm_day_before'] = storm['storm_day_before'].map({0: 'No Storm', 1: 'Storm Day Before'})

print("Storm impact on no-shows:")
for _, row in storm.iterrows():
    print(f"  {row['storm_day_before']:<25} No-show rate: {row['rate']}%  ({row['no_shows']:,} / {row['total']:,})")


Storm impact on no-shows:
  No Storm                  No-show rate: 20.7%  (139 / 670)
  Storm Day Before          No-show rate: 9.9%  (4,539 / 46,002)


## No-Show by City


In [23]:
# === NO-SHOW BY CITY ===
city_ns = df.groupby('city').agg(
    total=('no_show_binary', 'count'),
    no_shows=('no_show_binary', 'sum')
).reset_index()
city_ns['rate'] = (city_ns['no_shows'] / city_ns['total'] * 100).round(1)
city_ns = city_ns.sort_values('no_shows', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Showed Up', y=city_ns['city'], x=city_ns['total'] - city_ns['no_shows'],
                     orientation='h', marker_color='#2A9D8F'))
fig.add_trace(go.Bar(name='No-Show', y=city_ns['city'], x=city_ns['no_shows'],
                     orientation='h', marker_color='#E76F51'))
fig.update_layout(barmode='stack', title='Appointments by City (Show vs No-Show)',
                  height=450, yaxis={'categoryorder': 'total ascending'})
fig.show()


## 💰 Estimated Business Loss

Estimating the financial impact of no-shows using average consultation costs per specialty.


In [24]:
# === ESTIMATED BUSINESS LOSS ===
# Average consultation cost estimates by specialty (in USD)
cost_estimates = {
    'physiotherapy': 75,
    'psychotherapy': 120,
    'speech therapy': 100,
    'occupational therapy': 90,
    'enf': 60,
    'psychology': 120,
    'social service': 50,
    'nutrition': 70,
}

no_show_df = df[df['no_show'] == 'yes'].copy()
no_show_df['est_cost'] = no_show_df['specialty'].map(cost_estimates).fillna(80)  # default $80

total_loss = no_show_df['est_cost'].sum()
loss_by_spec = no_show_df.groupby('specialty')['est_cost'].agg(['sum', 'count']).reset_index()
loss_by_spec.columns = ['Specialty', 'Total Loss ($)', 'No-Shows']
loss_by_spec = loss_by_spec.sort_values('Total Loss ($)', ascending=False)

print("=" * 60)
print("   💰 ESTIMATED BUSINESS LOSS FROM NO-SHOWS")
print("=" * 60)
print(f"   Total No-Shows:          {len(no_show_df):>10,}")
print(f"   Estimated Total Loss:    ${total_loss:>10,.0f}")
print(f"   Avg Loss per No-Show:    ${total_loss/len(no_show_df):>10,.2f}")
print("=" * 60)

fig = px.bar(
    loss_by_spec, x='Specialty', y='Total Loss ($)',
    title=f'Estimated Revenue Loss by Specialty (Total: ${total_loss:,.0f})',
    color='Total Loss ($)', color_continuous_scale='Reds',
    text='Total Loss ($)'
)
fig.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig.update_layout(height=450, coloraxis_showscale=False, yaxis_tickprefix='$')
fig.show()

print("\nBreakdown:")
for _, row in loss_by_spec.iterrows():
    print(f"  {row['Specialty']:<25} {row['No-Shows']:>5,} no-shows → ${row['Total Loss ($)']:>10,.0f}")


   💰 ESTIMATED BUSINESS LOSS FROM NO-SHOWS
   Total No-Shows:               4,678
   Estimated Total Loss:    $   439,430
   Avg Loss per No-Show:    $     93.94



Breakdown:
  psychotherapy             1,075 no-shows → $   129,000
  speech therapy            1,109 no-shows → $   110,900
  physiotherapy               930 no-shows → $    69,750
  occupational therapy        550 no-shows → $    49,500
  pedagogo                    154 no-shows → $    12,320
  enf                          42 no-shows → $     2,520
  assist                       13 no-shows → $     1,040
  sem especialidade            13 no-shows → $     1,040


## 📊 Correlation Heatmap

Which numeric factors are most correlated with no-shows?


In [25]:
# === CORRELATION WITH NO-SHOW ===
numeric_cols = ['no_show_binary', 'age', 'under_12_years_old', 'over_60_years_old',
                'patient_needs_companion', 'average_temp_day', 'average_rain_day',
                'max_temp_day', 'max_rain_day', 'rainy_day_before', 'storm_day_before']

corr = df[numeric_cols].corr()

fig = px.imshow(
    corr, text_auto='.2f', aspect='auto',
    title='Correlation Matrix — Factors vs No-Show',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1
)
fig.update_layout(height=600, width=700)
fig.show()

# Show correlations with no_show specifically
no_show_corr = corr['no_show_binary'].drop('no_show_binary').sort_values(key=abs, ascending=False)
print("Correlation with no-show (strongest first):")
for col, val in no_show_corr.items():
    direction = "↑" if val > 0 else "↓"
    print(f"  {col:<30} {val:>+.4f}  {direction}")


Correlation with no-show (strongest first):
  storm_day_before               -0.0431  ↓
  rainy_day_before               -0.0431  ↓
  under_12_years_old             -0.0374  ↓
  patient_needs_companion        -0.0373  ↓
  average_temp_day               +0.0232  ↑
  average_rain_day               +0.0202  ↑
  max_rain_day                   +0.0147  ↑
  age                            +0.0145  ↑
  max_temp_day                   +0.0044  ↑
  over_60_years_old              +0.0006  ↑
